In [2]:
"""
PbS Quantum Dot Calculator
--------------------------
Calculates diameter, radius, molar extinction coefficient,
cuvette concentration, and stock concentration from:
  - 1S peak wavelength (nm)
  - Absorbance at 1S peak
  - Dilution factor
  - Path length (cm)

Formulas:
  Eg       : 1240 / peak_nm  (eV)
  Diameter : Eg = 0.41 + 1 / (0.0392*d^2 + 0.114*d)  → solved via quadratic
  epsilon  : 19600 * r^2.32  (M^-1 cm^-1),  r in nm
  Conc.    : Beer-Lambert  C = A / (epsilon * l)
"""

import math
import tkinter as tk
from tkinter import ttk, messagebox


# ── core calculations ────────────────────────────────────────────────────────

def calculate_diameter(peak_nm: float) -> float:
    """
    Solve for diameter d (nm) from:
        Eg = 0.41 + 1 / (0.0392*d^2 + 0.114*d)
    Rearranges to quadratic:
        0.0392*d^2 + 0.114*d - 1/(Eg-0.41) = 0
    """
    Eg = 1240.0 / peak_nm
    if Eg <= 0.41:
        raise ValueError(
            f"Eg ({Eg:.4f} eV) must be greater than 0.41 eV. "
            "Check your 1S peak wavelength (must be < 3024 nm)."
        )
    a = 0.0392
    b = 0.114
    c = -1.0 / (Eg - 0.41)
    discriminant = b**2 - 4 * a * c
    if discriminant < 0:
        raise ValueError("No real solution for diameter. Check your 1S peak wavelength.")
    d = (-b + math.sqrt(discriminant)) / (2 * a)
    if d <= 0:
        raise ValueError("Calculated diameter is non-positive. Check your 1S peak wavelength.")
    return d


def calculate_epsilon(radius_nm: float) -> float:
    """Return molar extinction coefficient (M^-1 cm^-1)."""
    return 19600.0 * (radius_nm ** 2.32)


def calculate_concentration(absorbance: float, epsilon: float,
                             path_length_cm: float) -> float:
    """Return concentration in mol/L via Beer-Lambert law."""
    return absorbance / (epsilon * path_length_cm)


def run_calculation(peak_nm: float, absorbance: float,
                    dilution_factor: float, path_length_cm: float) -> dict:
    """Run all calculations and return a results dict."""
    diameter  = calculate_diameter(peak_nm)
    radius    = diameter / 2.0
    epsilon   = calculate_epsilon(radius)
    c_cuvette = calculate_concentration(absorbance, epsilon, path_length_cm)
    c_stock   = c_cuvette * dilution_factor

    return {
        "Eg_eV"        : 1240.0 / peak_nm,
        "diameter_nm"  : diameter,
        "radius_nm"    : radius,
        "epsilon"      : epsilon,
        "c_cuvette_M"  : c_cuvette,
        "c_stock_M"    : c_stock,
    }


# ── CLI helper ───────────────────────────────────────────────────────────────

def cli_mode():
    print("\n── PbS QD Calculator ──")
    try:
        peak   = float(input("1S peak wavelength (nm)        : "))
        absorb = float(input("Absorbance at 1S peak          : "))
        dilut  = float(input("Dilution factor                : "))
        path   = float(input("Path length (cm) [default 1]   : ") or 1)
    except ValueError:
        print("Error: please enter numeric values.")
        return

    try:
        r = run_calculation(peak, absorb, dilut, path)
    except ValueError as e:
        print(f"\nError: {e}")
        return

    print("\n── Results ──────────────────────────────")
    print(f"  Bandgap energy (Eg)    : {r['Eg_eV']:.4f} eV")
    print(f"  Diameter               : {r['diameter_nm']:.4f} nm")
    print(f"  Radius                 : {r['radius_nm']:.4f} nm")
    print(f"  ε at 1S peak           : {r['epsilon']:.4e} M⁻¹cm⁻¹")
    print(f"  Cuvette concentration  : {r['c_cuvette_M']:.4e} M")
    print(f"  Stock concentration    : {r['c_stock_M']:.4e} M")
    print("─────────────────────────────────────────\n")


# ── GUI ──────────────────────────────────────────────────────────────────────

class QDCalculatorApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("PbS QD Calculator")
        self.resizable(False, False)
        self._build_ui()

    def _build_ui(self):
        pad = {"padx": 12, "pady": 6}

        tk.Label(self, text="PbS Quantum Dot Calculator",
                 font=("Helvetica", 15, "bold")).grid(
            row=0, column=0, columnspan=2, pady=(18, 4))

        tk.Label(self,
                 text="Eg = 0.41 + 1/(0.0392d² + 0.114d)   ·   ε₁s = 19600·r²·³²   ·   Beer–Lambert",
                 font=("Helvetica", 8), fg="gray").grid(
            row=1, column=0, columnspan=2, pady=(0, 12))

        ttk.Separator(self, orient="horizontal").grid(
            row=2, column=0, columnspan=2, sticky="ew", padx=12, pady=4)

        tk.Label(self, text="Inputs", font=("Helvetica", 10, "bold")).grid(
            row=3, column=0, columnspan=2, sticky="w", padx=14, pady=(8, 2))

        fields = [
            ("1S Peak (nm)",          "1255", "peak"),
            ("Absorbance at 1S peak", "0.35", "absorb"),
            ("Dilution factor",       "1",    "dilution"),
            ("Path length (cm)",      "1",    "path"),
        ]
        self.vars = {}
        for i, (label, default, key) in enumerate(fields):
            tk.Label(self, text=label, anchor="w").grid(
                row=i+4, column=0, sticky="w", **pad)
            var = tk.StringVar(value=default)
            self.vars[key] = var
            tk.Entry(self, textvariable=var, width=18).grid(
                row=i+4, column=1, sticky="ew", **pad)

        ttk.Separator(self, orient="horizontal").grid(
            row=8, column=0, columnspan=2, sticky="ew", padx=12, pady=8)

        tk.Button(self, text="Calculate", command=self._calculate,
                  bg="#2563EB", fg="white",
                  font=("Helvetica", 11, "bold"),
                  relief="flat", padx=18, pady=6, cursor="hand2").grid(
            row=9, column=0, columnspan=2, pady=(0, 10))

        ttk.Separator(self, orient="horizontal").grid(
            row=10, column=0, columnspan=2, sticky="ew", padx=12, pady=4)

        tk.Label(self, text="Results", font=("Helvetica", 10, "bold")).grid(
            row=11, column=0, columnspan=2, sticky="w", padx=14, pady=(6, 2))

        result_fields = [
            ("Bandgap energy Eg (eV)",     "res_eg"),
            ("Diameter (nm)",              "res_d"),
            ("Radius (nm)",                "res_r"),
            ("ε₁s (M⁻¹cm⁻¹)",             "res_eps"),
            ("Cuvette concentration (M)",  "res_cuvette"),
            ("Stock concentration (M)",    "res_stock"),
        ]
        self.result_vars = {}
        for i, (label, key) in enumerate(result_fields):
            tk.Label(self, text=label, anchor="w", fg="gray").grid(
                row=i+12, column=0, sticky="w", **pad)
            var = tk.StringVar(value="—")
            self.result_vars[key] = var
            tk.Label(self, textvariable=var, anchor="e",
                     font=("Courier", 11), fg="#1D4ED8").grid(
                row=i+12, column=1, sticky="e", **pad)

        ttk.Separator(self, orient="horizontal").grid(
            row=18, column=0, columnspan=2, sticky="ew", padx=12, pady=(10, 4))

        tk.Label(self,
                 text="C_stock = C_cuvette × dilution factor",
                 font=("Helvetica", 8), fg="gray").grid(
            row=19, column=0, columnspan=2, pady=(2, 16))

    def _calculate(self):
        try:
            peak   = float(self.vars["peak"].get())
            absorb = float(self.vars["absorb"].get())
            dilut  = float(self.vars["dilution"].get())
            path   = float(self.vars["path"].get())
        except ValueError:
            messagebox.showerror("Input error", "All fields must be numeric.")
            return

        if peak <= 0 or absorb <= 0 or dilut <= 0 or path <= 0:
            messagebox.showerror("Input error", "All values must be positive.")
            return

        try:
            r = run_calculation(peak, absorb, dilut, path)
        except ValueError as e:
            messagebox.showerror("Calculation error", str(e))
            return

        self.result_vars["res_eg"].set(f"{r['Eg_eV']:.4f}")
        self.result_vars["res_d"].set(f"{r['diameter_nm']:.4f}")
        self.result_vars["res_r"].set(f"{r['radius_nm']:.4f}")
        self.result_vars["res_eps"].set(f"{r['epsilon']:.4e}")
        self.result_vars["res_cuvette"].set(f"{r['c_cuvette_M']:.4e}")
        self.result_vars["res_stock"].set(f"{r['c_stock_M']:.4e}")


# ── entry point ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import sys
    if "--cli" in sys.argv:
        cli_mode()
    else:
        app = QDCalculatorApp()
        app.mainloop()